In [3]:
import pandas as pd

labels = pd.read_csv(r'C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\mimic-cxr-2.0.0-chexpert.csv')

# Tórax completamente sano: No Finding = 1, todo lo demás NaN o 0
no_finding = labels[
    (labels['No Finding'] == 1) &
    (labels.drop(columns=['subject_id', 'study_id', 'No Finding'])
           .fillna(0).sum(axis=1) == 0)
]

# Muestra aleatoria de 150 estudios
muestra_sanos = no_finding.sample(n=150, random_state=42)

# Muestra aleatoria general (150 estudios de cualquier finding)
muestra_general = labels.sample(n=150, random_state=99)

todos = pd.concat([muestra_sanos, muestra_general]).drop_duplicates(subset='study_id')
print(f"Total estudios seleccionados: {len(todos)}")

Total estudios seleccionados: 300


In [5]:
# IMAGE_FILENAMES tiene formato: files/p10/p10000032/s50414267/02aa804e-bde0afdd-...jpg
filenames = pd.read_csv(r'C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\IMAGE_FILENAMES', header=None, names=['path'])

# Extraer study_id del path (el folder tipo s50414267)
filenames['study_id'] = filenames['path'].str.extract(r'/(s\d+)/').astype(str)
# Convertir a int para hacer merge
filenames['study_id'] = filenames['study_id'].str.replace('s','').astype(int)

seleccionadas = filenames[filenames['study_id'].isin(todos['study_id'])]
print(f"Total imágenes a descargar: {len(seleccionadas)}")

# Guardar lista para wget
seleccionadas['path'].to_csv('MI_SUBSET.txt', index=False, header=False)

Total imágenes a descargar: 538


In [9]:
# Cargar metadatos con ViewPosition
metadata = pd.read_csv(r'C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\mimic-cxr-2.0.0-metadata.csv')

# metadata tiene: dicom_id, subject_id, study_id, ViewPosition, etc.
# Filtrar PA y AP
metadata_pa_ap = metadata[metadata['ViewPosition'].isin(['PA', 'AP'])][['dicom_id', 'study_id']]

# Reconstruir seleccionadas
seleccionadas = filenames[filenames['study_id'].isin(todos['study_id'])].copy()

# Merge
seleccionadas_pa_ap = seleccionadas.merge(metadata_pa_ap, on=['dicom_id', 'study_id'], how='inner')

print(f"Imágenes totales seleccionadas: {len(seleccionadas)}")
print(f"  PA: {len(seleccionadas.merge(metadata[metadata['ViewPosition']=='PA'][['dicom_id','study_id']], on=['dicom_id','study_id'], how='inner'))}")
print(f"  AP: {len(seleccionadas.merge(metadata[metadata['ViewPosition']=='AP'][['dicom_id','study_id']], on=['dicom_id','study_id'], how='inner'))}")
print(f"  PA+AP total: {len(seleccionadas_pa_ap)}")

# Guardar
seleccionadas_pa_ap['path'].to_csv('MI_SUBSET_PA_AP.txt', index=False, header=False)
print("✓ Guardado MI_SUBSET_PA_AP.txt")

Imágenes totales seleccionadas: 538
  PA: 170
  AP: 145
  PA+AP total: 315
✓ Guardado MI_SUBSET_PA_AP.txt


In [13]:
import requests
from pathlib import Path

USUARIO = "trodriguez"
PASSWORD = "Lucho78963"  # la ingresas tú mismo
BASE_URL = "https://physionet.org/files/mimic-cxr-jpg/2.1.0/"
DEST_DIR = r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset_prueba\subset_pa_ap"

with open('MI_SUBSET_PA_AP.txt') as f:
    paths = [line.strip() for line in f if line.strip()]

Path(DEST_DIR).mkdir(parents=True, exist_ok=True)

for i, rel_path in enumerate(paths[:3]):
    url = BASE_URL + rel_path.replace('files/', '', 1)
    dest = Path(DEST_DIR) / Path(rel_path).name
    
    if dest.exists():  # saltar si ya existe
        continue
    
    r = requests.get(url, auth=(USUARIO, PASSWORD))
    if r.status_code == 200:
        dest.write_bytes(r.content)
        print(f"[{i+1}/{len(paths)}] ✓ {Path(rel_path).name}")
    else:
        print(f"[{i+1}/{len(paths)}] ✗ Error {r.status_code}: {rel_path}")

[1/315] ✗ Error 403: files/p10/p10003400/s50805197/fc09630f-022d31c2-408b148f-95c499df-e8a6b18e.jpg
[2/315] ✗ Error 403: files/p10/p10026255/s52168916/d64c5ade-d7914a60-c528f838-1724e63d-a5e43b4e.jpg
[3/315] ✗ Error 403: files/p10/p10026255/s52168916/dd966a51-a5487743-b2830c8e-434aaa64-f038d753.jpg


In [14]:
# Solo imprime las primeras 3 URLs sin descargar nada
with open('MI_SUBSET_PA_AP.txt') as f:
    paths = [line.strip() for line in f if line.strip()]

for p in paths[:3]:
    print(BASE_URL + p)

https://physionet.org/files/mimic-cxr-jpg/2.1.0/files/p10/p10003400/s50805197/fc09630f-022d31c2-408b148f-95c499df-e8a6b18e.jpg
https://physionet.org/files/mimic-cxr-jpg/2.1.0/files/p10/p10026255/s52168916/d64c5ade-d7914a60-c528f838-1724e63d-a5e43b4e.jpg
https://physionet.org/files/mimic-cxr-jpg/2.1.0/files/p10/p10026255/s52168916/dd966a51-a5487743-b2830c8e-434aaa64-f038d753.jpg


In [20]:
import requests

USUARIO = "trodriguez"
TOKEN   = "tu_token_aqui"  # en lugar de la contraseña

url = "https://physionet.org/files/mimic-cxr-jpg/2.1.0/README"
r = requests.get(url, auth=(USUARIO, TOKEN))
print(f"Status: {r.status_code}")
print(r.text[:200])

Status: 403
<!DOCTYPE html>

<html lang="en">
  <head>
    <meta charset="UTF-8">
    <title>
403: Forbidden Access
</title>
    
    
<link rel="stylesheet" type="text/css" href="/static/bootstrap/css/bootstrap.


In [ ]:
wget -r -N -c -np -nH --cut-dirs=6 --user trodriguez --ask-password -i "C:\Users\trodr\Documents\proyecto-torax-v2.0\02-exploracion\MI_SUBSET_PA_AP.txt" --base=https://physionet.org/files/mimic-cxr-jpg/2.1.0/

In [ ]:
wget -r -N -c -np -nH --cut-dirs=3 --user trodriguez --ask-Lucho78963 -i "C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\subset500_Calidad.txt" --base=https://physionet.org/files/mimic-cxr-jpg/2.1.0/

In [ ]:
wget.exe -r -N -c -np -nH --cut-dirs=3 --user trodriguez `
  -i "C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\subset3000_descarga.txt" `
  --base=https://physionet.org/files/mimic-cxr-jpg/2.1.0/ `
  -P "C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\3000imagenes"

wget.exe -r -N -c -np -nH --cut-dirs=3 --user trodriguez --ask-password -i "C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\subset3000_descarga.txt" --base=https://physionet.org/files/mimic-cxr-jpg/2.1.0/ -P "C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\3000imagenes"